# Session 23 — Automated ML Deployment using BentoML and Docker

**Goal:** take a trained scikit-learn classifier and turn it into a running,
containerized REST service *without hand-writing the serving layer* — save the model
to BentoML's model store, declare a service, build a **Bento**, let BentoML generate
the Dockerfile and image, run the container, and call its endpoint.

## What BentoML automates

Session 6 (`Containerized ML Apps with Docker, Flask-FastAPI, and Kubernetes on GCP`)
and Session 7 (`Developing and Deploying APIs for ML Models`) build the serving path
by hand: you write the FastAPI app, the request/response models, the `joblib.load`
call, the `requirements.txt`, the `Dockerfile`, the `CMD uvicorn ...` line, and you
keep all of them in sync with the model you actually trained.

BentoML collapses that into two artifacts you *do* have to write — a `service.py` and
a `bentofile.yaml` — and generates the rest. It versions the model, records the Python
and library versions used at training time, generates the Dockerfile, wires up an
OpenAPI-documented HTTP server with request validation and adaptive batching, and
packages model + code + dependencies into one immutable, addressable unit (the Bento).

The trade is the same as every automation trade in this course: you give up direct
control of the serving stack in exchange for not maintaining it. When you need
something BentoML doesn't express, you drop back to Session 6's hand-written approach.

## The dataset

This session uses the UCI **Glass Identification** dataset (id 42) — 214 real samples
of glass fragments from forensic casework, described by refractive index and the
weight-percent of eight oxides (Na, Mg, Al, Si, K, Ca, Ba, Fe), with a 6-class target
(`Type_of_glass`: window glass of several manufacturing types, containers, tableware,
headlamps).

It's a deliberately *small* dataset, and that's the point for this session: the model
trains in under a second, so nothing about the notebook is waiting on training — the
interesting time is spent on packaging, image build, and the round trip to the
container. The multi-class target also makes the prediction response more informative
than a binary yes/no when you inspect it over HTTP, and nine plain float features keep
the request payload readable by hand.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe* says
exactly what to look at in that cell's output; *Infer* says what conclusion that output
should lead you to, and what it would mean if you saw something different instead.
Packaging pipelines fail quietly — a stale model tag or a missing dependency in
`bentofile.yaml` doesn't error at build time, it errors *inside the container* minutes
later — so treat the notes as a checklist and stop when one doesn't match.

## Prerequisites

You need a working **Docker daemon** (Docker Desktop, Colima, or a Linux docker engine)
in addition to the Python packages. `bentoml containerize` shells out to `docker build`,
so without a running daemon everything up to Step 6 works and Step 7 fails.

```bash
pip install bentoml scikit-learn pandas ucimlrepo requests
docker version   # must print both Client and Server sections
```

## Step 1 — Fetch the dataset

Fetching from the UCI ML Repository directly keeps this notebook runnable by anyone
rather than depending on a CSV already sitting on your machine.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

glass = fetch_ucirepo(id=42)
X = glass.data.features
y = glass.data.targets["Type_of_glass"]

print(f"{X.shape[0]} rows, {X.shape[1]} features")
print(X.columns.tolist())
print(y.value_counts().sort_index())

**Observe:** the shape line (`214 rows, 9 features`), the feature list
`['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']`, and the class counts —
`1: 70, 2: 76, 3: 17, 5: 13, 6: 9, 7: 29`.
**Infer:** two things matter downstream. First, the class labels are **not**
`0..5` — they are `1, 2, 3, 5, 6, 7`, with `4` absent from the data entirely; the
model will predict those literal labels, so a response containing a `4` would mean
something re-encoded the classes behind your back. Second, the rare classes (9 and
13 samples) mean any single train/test split is noisy — that's fine for a packaging
demo but is exactly the kind of detail you would *not* wave away if this were a
model you intended to actually deploy.

## Step 2 — Train the model

Nothing novel here — this is the same shape of code as Session 1's training step. The
only choice that matters for the rest of the notebook is that we keep the feature
*order* explicit, because the service will need to reconstruct it from JSON.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

FEATURES = ["RI", "Na", "Mg", "Al", "Si", "K", "Ca", "Ba", "Fe"]

X_train, X_test, y_train, y_test = train_test_split(
    X[FEATURES], y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=300, random_state=42)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average="macro")
print(f"accuracy       : {acc:.3f}")
print(f"macro F1       : {f1:.3f}")
print(f"classes seen   : {clf.classes_.tolist()}")

**Observe:** the three printed lines — a real run gives
`accuracy: 0.796`, `macro F1: 0.677`, and
`classes seen: [1, 2, 3, 5, 6, 7]`.
**Infer:** the gap between accuracy (0.796) and macro F1 (0.677) is the rare classes
showing up in the metrics: the model does well on types 1 and 2 (which are most of the
data) and poorly on types 3 and 6 (two to four test samples each). That's expected and
not a blocker here. The line that *is* load-bearing for this session is
`classes seen` — those six labels are what the deployed endpoint will return, and
`clf.classes_` is the only authoritative source for their order when you later read
`predict_proba` output. If this list showed only five entries, `stratify=y` failed to
place a rare class in the training half and the packaged model would be structurally
incapable of ever predicting it.

## Step 3 — Save the model into the BentoML model store

`bentoml.sklearn.save_model` is not `joblib.dump`. It writes the model into a local,
content-addressed store under `~/bentoml/models/`, assigns it an immutable version tag,
and records the Python version and the versions of scikit-learn/numpy present *right
now* — so the image built in Step 7 installs the same ones rather than whatever pip
resolves that day.

`signatures` declares which methods are exposed for serving and whether BentoML may
batch concurrent requests into a single call. `batchable=True` on `predict` is free
throughput for a vectorized sklearn model.

In [ ]:
import bentoml

saved_model = bentoml.sklearn.save_model(
    "glass_clf",
    clf,
    signatures={
        "predict":       {"batchable": True, "batch_dim": 0},
        "predict_proba": {"batchable": True, "batch_dim": 0},
    },
    labels={"owner": "mlops-course", "stage": "demo"},
    metadata={
        "dataset": "uci-glass-identification-42",
        "accuracy": round(float(acc), 4),
        "macro_f1": round(float(f1), 4),
    },
    custom_objects={"features": FEATURES, "classes": clf.classes_.tolist()},
)
print(saved_model)

**Observe:** the printed model object, e.g.
`Model(tag="glass_clf:kx3n7qtfp2abc4de", path="~/bentoml/models/glass_clf/kx3n7qtfp2abc4de/")`.
**Infer:** the random-looking suffix after the colon is the **version tag**, and it is
new on every single call to `save_model` — re-running this cell does not overwrite the
previous model, it adds another version and moves the `latest` alias. That is the
mechanism that makes a Bento reproducible, and it is also the single most common source
of confusion in this workflow: if you retrain and forget to rebuild, the container keeps
serving the older version quite happily and nothing errors.

Note what went into `custom_objects`: the feature order and the class list. Those travel
*with the model*, so the service in Step 4 doesn't have to hard-code them and can never
drift out of sync with the model it is serving.

In [ ]:
!bentoml models list glass_clf

**Observe:** the table — `Tag`, `Module`, `Size`, `Creation Time` —
with a row like `glass_clf:kx3n7qtfp2abc4de   bentoml.sklearn   1.87 MiB   2026-08-25 10:14:02`.
**Infer:** one row means one save; if you see several rows, you have re-run Step 3 and
must be deliberate about which tag you build against. `Module: bentoml.sklearn` confirms
BentoML recorded the *framework*, not just an opaque pickle — that is what lets it load
the model correctly inside the container without your service code doing any
deserialization work. A `Size` in the low MiB is right for a 300-tree forest on 214
rows; a size in the hundreds of KiB would suggest the model didn't actually fit.

## Step 4 — Write the service definition

This is the file Session 6 and 7 would spend most of their length on. Here it is about
25 lines, and note what is *absent*: no `uvicorn` invocation, no route decorators with
paths and methods, no manual JSON parsing, no `joblib.load`, no error handling for a
malformed body. `@bentoml.service` supplies the server; `@bentoml.api` turns a typed
Python method into a validated HTTP endpoint with an OpenAPI schema.

The type annotations are the contract: `dict[str, float]` on the input means BentoML
rejects a request with a missing or non-numeric field with a `400` and a readable
message, before your code runs.

In [ ]:
%%writefile service.py
import bentoml
import numpy as np

glass_ref = bentoml.models.get("glass_clf:latest")
FEATURES = glass_ref.custom_objects["features"]
CLASSES = glass_ref.custom_objects["classes"]


@bentoml.service(
    name="glass_classifier",
    resources={"cpu": "1"},
    traffic={"timeout": 10},
)
class GlassClassifier:
    bento_model = glass_ref

    def __init__(self) -> None:
        self.model = bentoml.sklearn.load_model(self.bento_model)

    @bentoml.api
    def classify(self, sample: dict[str, float]) -> dict:
        row = np.array([[sample[f] for f in FEATURES]])
        proba = self.model.predict_proba(row)[0]
        best = int(np.argmax(proba))
        return {
            "predicted_type": int(CLASSES[best]),
            "confidence": round(float(proba[best]), 4),
            "scores": {str(c): round(float(p), 4) for c, p in zip(CLASSES, proba)},
            "model_version": str(self.bento_model.tag),
        }

**Observe:** the single line `Writing service.py`.
**Infer:** `%%writefile` overwrites silently — if you edit this cell and re-run it, the
file changes but any server already running from Step 5 keeps the old code in memory
unless it was started with `--reload`. Two details in the file are worth pausing on:
`bentoml.models.get("glass_clf:latest")` resolves the alias **at build time**, so the
Bento pins whatever concrete version `latest` pointed at when you built — the image is
never ambiguous even though this line is. And returning `model_version` in the response
body means every prediction is self-identifying; when a downstream consumer reports a
weird answer, you can tell immediately which model produced it instead of guessing from
deployment timestamps.

## Step 5 — Serve it locally, before building anything

Always run the service outside a container first. A bug in `service.py` found here costs
seconds; the same bug found after a two-minute image build costs the build.

In [ ]:
# Run this in a terminal (it blocks) -- or with `&` from the notebook:
!bentoml serve service:GlassClassifier --port 3000 --reload

**Observe:** the startup log —
`Starting dev BentoServer from "service:GlassClassifier" listening on http://localhost:3000`,
followed by a `Service glass_classifier initialized` line and the Uvicorn banner.
**Infer:** reaching the "listening on" line means the module imported cleanly, which in
turn means `bentoml.models.get("glass_clf:latest")` found the model saved in Step 3 —
service startup *is* the model-resolution check. The most common failure here is
`bentoml.exceptions.NotFound: Model 'glass_clf:latest' is not found in BentoML store`,
which means Step 3 ran in a different environment (or a different `BENTOML_HOME`) than
this server. Visit `http://localhost:3000` in a browser and you get a Swagger UI for
`/classify` that you never wrote — that page is generated from the type annotations.

In [ ]:
import requests

sample = {
    "RI": 1.51761, "Na": 13.89, "Mg": 3.60, "Al": 1.36, "Si": 72.73,
    "K": 0.48, "Ca": 7.83, "Ba": 0.0, "Fe": 0.0,
}

resp = requests.post("http://localhost:3000/classify", json={"sample": sample}, timeout=10)
print(resp.status_code)
print(resp.json())

**Observe:** status `200` and a body like
`{'predicted_type': 1, 'confidence': 0.8433, 'scores': {'1': 0.8433, '2': 0.1367, '3': 0.02, '5': 0.0, '6': 0.0, '7': 0.0}, 'model_version': 'glass_clf:kx3n7qtfp2abc4de'}`.
**Infer:** the request body is `{"sample": {...}}`, not `{...}` — BentoML keys the JSON
by *parameter name*, so a flat payload returns `422 Unprocessable Entity` with
`field required: sample`. That mapping is the one piece of BentoML's convention you have
to learn; everything else follows from the annotations. On the content side, type 1
(float-processed building window glass) at 84% for a sample with high magnesium and no
barium is chemically sensible. Confirm `model_version` matches the tag from Step 3 — if
it doesn't, `latest` is pointing at an older save and Step 7 would package the wrong
model.

## Step 6 — Declare the Bento with `bentofile.yaml`

A **Bento** is the deployable unit: your service code, the model from the store, the
dependency list, and the Python version, in one versioned directory. This file says what
goes in. `include` keeps the archive minimal — without it, BentoML sweeps in the whole
working directory, notebooks and datasets included.

You do **not** list the model here. BentoML traces it from `bentoml.models.get(...)` in
`service.py` and pins the resolved version automatically.

In [ ]:
%%writefile bentofile.yaml
service: "service:GlassClassifier"
labels:
  owner: mlops-course
  session: "23"
include:
  - "service.py"
python:
  packages:
    - scikit-learn==1.5.2
    - numpy==2.1.1
docker:
  python_version: "3.11"

**Observe:** `Writing bentofile.yaml`, and re-read the two pinned
package versions against what you actually have installed (`pip show scikit-learn`).
**Infer:** these pins are the highest-risk lines in the whole notebook. If the version
you pin here differs from the version that produced the model in Step 3, the container
will load a pickle written by a different scikit-learn — sometimes that raises
`InconsistentVersionWarning` and works, sometimes it raises `AttributeError` deep inside
an estimator's `__setstate__`, and occasionally it loads and returns subtly wrong
numbers. Prefer generating this section with `bentoml build` in an environment that
matches training, or read the recorded versions back with
`bentoml models get glass_clf:latest` and copy them here verbatim.

## Step 7 — Build the Bento

In [ ]:
!bentoml build

**Observe:** the ASCII BentoML banner, then
`Building BentoML service "glass_classifier:d7pqz2vfr4xyzabc" from build context "/Users/you/session23"`,
a `Packing model "glass_clf:kx3n7qtfp2abc4de"` line, and finally
`Successfully built Bento(tag="glass_classifier:d7pqz2vfr4xyzabc")`.
**Infer:** the `Packing model` line is the one to actually read — it prints the concrete
model version that got frozen into this Bento, and it should match the tag from Step 3
character for character. This is where "I retrained but forgot to rebuild" gets caught,
and it is the only place it gets caught cheaply. Note also that the Bento tag
(`glass_classifier:d7pq...`) is a *different* identifier from the model tag; one Bento
version wraps one model version, and both are immutable.

In [ ]:
!bentoml list
!bentoml get glass_classifier:latest

**Observe:** the `bentoml list` row for `glass_classifier`, then the
detailed YAML from `bentoml get` — specifically the `models:` block listing
`- tag: glass_clf:kx3n7qtfp2abc4de` and the `python: packages:` block echoing your pins.
**Infer:** this YAML *is* the manifest that will be materialized inside the image, so
anything wrong here is wrong in the container. Two checks pay for themselves: the
`models:` list has exactly one entry (more than one means `service.py` is referencing a
model you forgot about), and `include` didn't drag in extra files (look at the `Size`
in `bentoml list` — a few MiB is right; hundreds of MiB means the build context swept up
your data directory and the image build will be correspondingly slow).

## Step 8 — Containerize

`bentoml containerize` generates a Dockerfile from the manifest above and hands it to
`docker build`. You never write or maintain the Dockerfile — this is the step that
replaces the hand-written one in Session 6.

In [ ]:
!bentoml containerize glass_classifier:latest -t glass-classifier:0.1.0

**Observe:** `Building OCI-compliant image for glass_classifier:d7pqz2vfr4xyzabc with docker`,
then the ordinary `docker build` layer output (`[+] Building 96.4s (18/18) FINISHED`),
and finally
`Successfully built Bento container for "glass_classifier:latest" with tag(s) "glass-classifier:0.1.0"`.
**Infer:** the layer count and the base-image pull tell you what BentoML generated:
a slim Python 3.11 base, an `apt` layer, a pip layer for your pinned packages, and a
COPY of the Bento. The pip layer is the slow one on a cold cache and is cached on
rebuilds — so a *second* containerize after only changing `service.py` should finish in
seconds. If it doesn't, your `bentofile.yaml` changed too and every layer after it was
invalidated.

### If this step fails

Three failures are common enough to name.

**`docker daemon is not running` / `Cannot connect to the Docker daemon at unix:///var/run/docker.sock`**
— BentoML found the `docker` binary but not a live engine. Start Docker Desktop (or
`colima start`) and re-run; nothing before this step needs redoing, the Bento is already
built and sitting in the store.

**`exec format error` when the image later runs on a cloud host.** This one does *not*
fail at build time, which is what makes it worth knowing in advance. On an Apple Silicon
machine, `bentoml containerize` builds `linux/arm64` by default; push that to an x86
Kubernetes node (as in Session 6) and the container crash-loops with

```
exec /usr/bin/bento: exec format error
```

**Observe:** whether `docker image inspect glass-classifier:0.1.0 --format '{{.Architecture}}'`
prints `arm64` while your target cluster is `amd64`.
**Infer:** this is an architecture mismatch, not a BentoML or model problem — the image
is fine, it is just built for the wrong CPU. The fix is to build explicitly for the
target rather than the host:

```bash
bentoml containerize glass_classifier:latest \
    -t glass-classifier:0.1.0 --opt platform=linux/amd64
```

That routes through `buildx` and emulation, so expect it to take noticeably longer than
the native build. Building for the wrong architecture and discovering it in the cluster
is the single most expensive version of this mistake, because the build succeeded and
the failure surfaces in someone else's environment.

**`no space left on device` mid-build** — accumulated Bento images are large. `docker
system prune -f` and re-run; the pip layer will rebuild from scratch.

## Step 9 — Run the container and call it

Same service, same endpoint, now with the dependencies and Python version frozen inside
the image instead of borrowed from your laptop.

In [ ]:
!docker run -d --rm -p 3000:3000 --name glass-svc glass-classifier:0.1.0
!sleep 5 && docker logs glass-svc | tail -n 5

**Observe:** the 64-character container ID from `docker run`, then in
the logs `Starting production HTTP BentoServer from "." listening on http://0.0.0.0:3000`
and a `Service glass_classifier initialized` line.
**Infer:** `0.0.0.0` rather than `localhost` is what makes the port mapping work — a
service bound to `127.0.0.1` *inside* a container is unreachable from the host no matter
what `-p` says, and BentoML's production server gets this right for you. If `docker logs`
shows the container exited instead, the two usual causes are the dependency mismatch from
Step 6 (a traceback ending in `ModuleNotFoundError` or an sklearn unpickling error) and
the architecture mismatch from Step 8 (`exec format error`) — the log line distinguishes
them immediately.

In [ ]:
resp = requests.post("http://localhost:3000/classify", json={"sample": sample}, timeout=10)
print(resp.status_code, resp.json())

# A different sample: high barium, low magnesium -- chemically a headlamp
headlamp = {
    "RI": 1.51653, "Na": 11.95, "Mg": 0.0, "Al": 1.19, "Si": 75.18,
    "K": 2.70, "Ca": 8.93, "Ba": 0.0, "Fe": 0.0,
}
print(requests.post("http://localhost:3000/classify", json={"sample": headlamp}, timeout=10).json())

**Observe:** the first response should be **byte-for-byte identical** to
the one you got from the local server in Step 5 — same `predicted_type`, same
`confidence`, same `model_version`. The second returns something like
`{'predicted_type': 7, 'confidence': 0.71, ...}`.
**Infer:** identical output from the local process and the container is the actual
success criterion of this whole session — it means the packaging step preserved the
model *and* its numerical behaviour, not just that a server came up. If the confidence
values differ even in the fourth decimal place, the container is running a different
scikit-learn or numpy build than your laptop, and Step 6's pins are wrong; treat that as
a failed deployment even though every command reported success. The second sample
predicting type 7 (headlamps) from zero magnesium and high potassium confirms the model
is discriminating on chemistry rather than defaulting to the majority class.

In [ ]:
!docker stop glass-svc

**Observe:** the container name `glass-svc` echoed back.
**Infer:** `--rm` in the `docker run` above means stopping also removes the container, so
`docker ps -a` should show nothing for `glass-svc`. The *image* remains — that is the
artifact you would push to a registry and hand to Session 6's Kubernetes manifests or
Session 14's deployment pipeline. Nothing about the Bento or the model store is affected
by stopping the container.

## Where this fits against Session 6

| | Session 6 (hand-written) | Session 23 (BentoML) |
|---|---|---|
| Serving app | you write FastAPI routes + Pydantic models | `@bentoml.api` on a typed method |
| Model loading | `joblib.load` at import time | resolved from the model store, version-pinned |
| Dependencies | you maintain `requirements.txt` | recorded at save time, declared in `bentofile.yaml` |
| Dockerfile | you write and maintain it | generated by `bentoml containerize` |
| Versioning | image tag only | model tag + Bento tag, both immutable |
| API docs | you add them | OpenAPI/Swagger generated from annotations |
| Batching | you implement it | `batchable=True` in `signatures` |

The output of both sessions is the same kind of thing: an OCI image exposing a REST
endpoint. Session 6 is worth doing first precisely because it shows you what BentoML is
doing on your behalf — and when a Bento misbehaves, debugging it means reasoning about
that same generated stack.

## What to try next

* Deploy this image with Session 6's Kubernetes manifests instead of `docker run` —
  the Bento image is an ordinary OCI image, so the only change is the container spec.
  Watch for the `linux/amd64` platform issue from Step 8 the moment it leaves your laptop.
* Add a second `@bentoml.api` method that accepts a *list* of samples and returns a list
  of predictions, then compare throughput against looping over the single-row endpoint —
  this is where the `batchable=True` signature from Step 3 starts to pay.
* Wire `bentoml build` and `bentoml containerize` into the GitHub Actions pipeline from
  Session 10, gated by the model-quality check from Session 24, so a Bento is only built
  when the candidate model passes.
* Compare this packaging story with Session 25's FastAPI deployment of a FLAML model:
  the same model, served two ways, is the clearest way to feel what BentoML actually saves.
* Register the model in MLflow (Session 1) *and* the BentoML store, and think about which
  one is the source of truth for "what is deployed" — running both without deciding is a
  reliable way to ship the wrong version.